In [1]:
import numpy as np 
import pandas as pd

import pickle

import nltk
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Movie Recommendation System (Content Based Filtering)

In [2]:
credits = pd.read_csv('credits.csv')
credits_main = credits.copy()
credits

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862
...,...,...,...
45471,"[{'cast_id': 0, 'character': '', 'credit_id': ...","[{'credit_id': '5894a97d925141426c00818c', 'de...",439050
45472,"[{'cast_id': 1002, 'character': 'Sister Angela...","[{'credit_id': '52fe4af1c3a36847f81e9b15', 'de...",111109
45473,"[{'cast_id': 6, 'character': 'Emily Shaw', 'cr...","[{'credit_id': '52fe4776c3a368484e0c8387', 'de...",67758
45474,"[{'cast_id': 2, 'character': '', 'credit_id': ...","[{'credit_id': '533bccebc3a36844cf0011a7', 'de...",227506


In [3]:
keywords = pd.read_csv('keywords.csv')
keywords_main = keywords.copy()
keywords

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."
...,...,...
46414,439050,"[{'id': 10703, 'name': 'tragic love'}]"
46415,111109,"[{'id': 2679, 'name': 'artist'}, {'id': 14531,..."
46416,67758,[]
46417,227506,[]


In [4]:
movies_metadata = pd.read_csv('movies_metadata.csv')
movies_metadata_main = movies_metadata.copy()
movies_metadata

C:\Users\Dines\AppData\Local\Temp\ipykernel_13276\1430886609.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_metadata = pd.read_csv('movies_metadata.csv')


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45461,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",http://www.imdb.com/title/tt6209470/,439050,tt6209470,fa,رگ خواب,Rising and falling between a man and woman.,...,NaN,0.0,90.0,"[{'iso_639_1': 'fa', 'name': 'فارسی'}]",Released,Rising and falling between a man and woman,Subdue,False,4.0,1.0
45462,False,NaN,0,"[{'id': 18, 'name': 'Drama'}]",NaN,111109,tt2028550,tl,Siglo ng Pagluluwal,An artist struggles to finish his work while a...,...,2011-11-17,0.0,360.0,"[{'iso_639_1': 'tl', 'name': ''}]",Released,NaN,Century of Birthing,False,9.0,3.0
45463,False,NaN,0,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",NaN,67758,tt0303758,en,Betrayal,"When one of her hits goes wrong, a professiona...",...,2003-08-01,0.0,90.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,A deadly game of wits.,Betrayal,False,3.8,6.0
45464,False,NaN,0,[],NaN,227506,tt0008536,en,Satana likuyushchiy,"In a small town live two brothers, one a minis...",...,1917-10-21,0.0,87.0,[],Released,NaN,Satan Triumphant,False,0.0,0.0


In [5]:
movies_metadata.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [6]:
drop_cols = ['adult', 'budget', 'popularity', 'production_countries','poster_path' ,'homepage', 'imdb_id', 'original_language', 'release_date','revenue','runtime','spoken_languages','status', 'original_title','video', 'vote_average','vote_count']

def remove_cols(drop_cols):
    lst = []
    for i in drop_cols:
        if i in movies_metadata.columns:
            lst.append(i)
        else:
            print(f"{i} is not present in the dataset")

    movies_metadata.drop(columns = lst, inplace = True)

remove_cols(drop_cols)

In [7]:
movies_metadata

,belongs_to_collection,genres,id,overview,production_companies,tagline,title
0,"{'id': 10194, 'name': 'Toy Story Collection', ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,"Led by Woody, Andy's toys live happily in his ...","[{'name': 'Pixar Animation Studios', 'id': 3}]",NaN,Toy Story
1,NaN,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,When siblings Judy and Peter discover an encha...,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...",Roll the dice and unleash the excitement!,Jumanji
2,"{'id': 119050, 'name': 'Grumpy Old Men Collect...","[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,A family wedding reignites the ancient feud be...,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...",Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men
3,NaN,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,"Cheated on, mistreated and stepped on, the wom...",[{'name': 'Twentieth Century Fox Film Corporat...,Friends are the people who let you be yourself...,Waiting to Exhale
4,"{'id': 96871, 'name': 'Father of the Bride Col...","[{'id': 35, 'name': 'Comedy'}]",11862,Just when George Banks has recovered from his ...,"[{'name': 'Sandollar Productions', 'id': 5842}...",Just When His World Is Back To Normal... He's ...,Father of the Bride Part II
...,...,...,...,...,...,...,...
45461,NaN,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",439050,Rising and falling between a man and woman.,[],Rising and falling between a man and woman,Subdue
45462,NaN,"[{'id': 18, 'name': 'Drama'}]",111109,An artist struggles to finish his work while a...,"[{'name': 'Sine Olivia', 'id': 19653}]",NaN,Century of Birthing
45463,NaN,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",67758,"When one of her hits goes wrong, a professiona...","[{'name': 'American World Pictures', 'id': 6165}]",A deadly game of wits.,Betrayal
45464,NaN,[],227506,"In a small town live two brothers, one a minis...","[{'name': 'Yermoliev', 'id': 88753}]",NaN,Satan Triumphant


In [8]:

movies_metadata['belongs_to_collection'].isnull().sum()
# since most of the rows of this column are null there is no particular use of it for our recommendation system
remove_cols(['belongs_to_collection'])

In [9]:
movies_metadata

,genres,id,overview,production_companies,tagline,title
0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,"Led by Woody, Andy's toys live happily in his ...","[{'name': 'Pixar Animation Studios', 'id': 3}]",NaN,Toy Story
1,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,When siblings Judy and Peter discover an encha...,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...",Roll the dice and unleash the excitement!,Jumanji
2,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,A family wedding reignites the ancient feud be...,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...",Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men
3,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,"Cheated on, mistreated and stepped on, the wom...",[{'name': 'Twentieth Century Fox Film Corporat...,Friends are the people who let you be yourself...,Waiting to Exhale
4,"[{'id': 35, 'name': 'Comedy'}]",11862,Just when George Banks has recovered from his ...,"[{'name': 'Sandollar Productions', 'id': 5842}...",Just When His World Is Back To Normal... He's ...,Father of the Bride Part II
...,...,...,...,...,...,...
45461,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",439050,Rising and falling between a man and woman.,[],Rising and falling between a man and woman,Subdue
45462,"[{'id': 18, 'name': 'Drama'}]",111109,An artist struggles to finish his work while a...,"[{'name': 'Sine Olivia', 'id': 19653}]",NaN,Century of Birthing
45463,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",67758,"When one of her hits goes wrong, a professiona...","[{'name': 'American World Pictures', 'id': 6165}]",A deadly game of wits.,Betrayal
45464,[],227506,"In a small town live two brothers, one a minis...","[{'name': 'Yermoliev', 'id': 88753}]",NaN,Satan Triumphant


In [10]:
for i in movies_metadata.columns:
    print(f'{i} column has {movies_metadata[i].isnull().sum()} null values')

genres column has 0 null values
id column has 0 null values
overview column has 954 null values
production_companies column has 3 null values
tagline column has 25054 null values
title column has 6 null values


In [11]:
remove_cols(['tagline'])
remove_cols(['production_companies'])

In [12]:
movies_metadata[movies_metadata['genres'] == '[]']

,genres,id,overview,title
55,[],124057,"Set in modern times, Alex finds King Arthur's ...",Kids of the Round Table
83,[],188588,"Filmed entirely on location in East Hampton, L...",Last Summer in the Hamptons
126,[],290157,"Michel Negroponte, a documentary filmmaker, me...",Jupiter's Wife
137,[],124639,A subtle yet violent commentary on feudal lords.,Target
390,[],267188,Jackie and Eugene are joined by a mystical win...,Desert Winds
...,...,...,...,...
45447,[],44324,The background of this picture represents a sc...,The Untameable Whiskers
45448,[],122036,This shows a prince entering upon the stage of...,The Imperceptable Transmutations
45455,[],67179,Sentenced to life imprisonment for illegal act...,St. Michael Had a Rooster
45464,[],227506,"In a small town live two brothers, one a minis...",Satan Triumphant


In [13]:
movies_metadata['id'] = pd.to_numeric(movies_metadata['id'], errors='coerce')
movies_metadata = movies_metadata.dropna(subset=['id'])

In [14]:
movies_metadata['id'] = movies_metadata['id'].astype('int64')

C:\Users\Dines\AppData\Local\Temp\ipykernel_13276\3157542909.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_metadata['id'] = movies_metadata['id'].astype('int64')


In [15]:
data = pd.merge(movies_metadata, keywords, on = 'id', how = 'inner')
data.head()

,genres,id,overview,title,keywords
0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,"Led by Woody, Andy's toys live happily in his ...",Toy Story,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,When siblings Judy and Peter discover an encha...,Jumanji,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,A family wedding reignites the ancient feud be...,Grumpier Old Men,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,"Cheated on, mistreated and stepped on, the wom...",Waiting to Exhale,"[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,"[{'id': 35, 'name': 'Comedy'}]",11862,Just when George Banks has recovered from his ...,Father of the Bride Part II,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."


In [16]:
data = pd.merge(data,credits ,on = 'id', how = 'inner')
data.head()

,genres,id,overview,title,keywords,cast,crew
0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,"Led by Woody, Andy's toys live happily in his ...",Toy Story,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,When siblings Judy and Peter discover an encha...,Jumanji,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,A family wedding reignites the ancient feud be...,Grumpier Old Men,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...","[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,"Cheated on, mistreated and stepped on, the wom...",Waiting to Exhale,"[{'id': 818, 'name': 'based on novel'}, {'id':...","[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,"[{'id': 35, 'name': 'Comedy'}]",11862,Just when George Banks has recovered from his ...,Father of the Bride Part II,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."


In [17]:
data.shape

(46628, 7)

In [18]:
data.isnull().sum()

genres        0
id            0
overview    995
title         4
keywords      0
cast          0
crew          0
dtype: int64

In [19]:
data.dropna(inplace = True)

In [20]:
data.isnull().sum()

genres      0
id          0
overview    0
title       0
keywords    0
cast        0
crew        0
dtype: int64

In [21]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45629 entries, 0 to 46627
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   genres    45629 non-null  object
 1   id        45629 non-null  int64 
 2   overview  45629 non-null  object
 3   title     45629 non-null  object
 4   keywords  45629 non-null  object
 5   cast      45629 non-null  object
 6   crew      45629 non-null  object
dtypes: int64(1), object(6)
memory usage: 2.8+ MB


In [22]:
data[data['id']== 468343]

,genres,id,overview,title,keywords,cast,crew
22027,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",468343,"In the 1910s, beautiful young Silja loses both...",Silja - nuorena nukkunut,[],[],"[{'credit_id': '597ae87a925141364e00a73e', 'de..."


In [23]:
type(data['genres'][0])

str

In [24]:
data.select_dtypes(include = 'object').columns

Index(['genres', 'overview', 'title', 'keywords', 'cast', 'crew'], dtype='object')

In [25]:
for i in data.select_dtypes(include = 'object').columns:
    data = data[data[i] != '[]']

In [26]:
data.shape

(30231, 7)

In [27]:
lst_cols = ['genres','keywords', 'cast', 'crew' ]
import ast
for col in lst_cols:
    data[col] = data[col].apply(ast.literal_eval)

In [28]:
for i in lst_cols:
    print(type(data[i][862]))

<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>


In [29]:
def clean_genres(dct):
    lst = []
    for i in dct:
        lst.append(i['name'])
    return lst

In [30]:
data['genres'] = data['genres'].apply(clean_genres)

In [31]:
def clean_keywords(dct):
    lst = []
    for i in dct:
        lst.append(i['name'])
    return lst

In [32]:
data['keywords'] =  data['keywords'].apply(clean_keywords)

In [33]:
def clean_cast(dct):
    lst = []
    counter = 0
    
    for i in dct:
        if counter == 3:
            break
        lst.append(i['name'])
        counter += 1
    
    return lst

In [34]:
data['cast'] = data['cast'].apply(clean_cast)

In [35]:
def clean_crew(dct):
    lst = []
    for i in dct:
        if i['job'] == 'Director':
            lst.append(i['name'])
    return lst

In [36]:
data['crew'] = data['crew'].apply(clean_crew)

In [37]:
data.head()

,genres,id,overview,title,keywords,cast,crew
0,"[Animation, Comedy, Family]",862,"Led by Woody, Andy's toys live happily in his ...",Toy Story,"[jealousy, toy, boy, friendship, friends, riva...","[Tom Hanks, Tim Allen, Don Rickles]",[John Lasseter]
1,"[Adventure, Fantasy, Family]",8844,When siblings Judy and Peter discover an encha...,Jumanji,"[board game, disappearance, based on children'...","[Robin Williams, Jonathan Hyde, Kirsten Dunst]",[Joe Johnston]
2,"[Romance, Comedy]",15602,A family wedding reignites the ancient feud be...,Grumpier Old Men,"[fishing, best friend, duringcreditsstinger, o...","[Walter Matthau, Jack Lemmon, Ann-Margret]",[Howard Deutch]
3,"[Comedy, Drama, Romance]",31357,"Cheated on, mistreated and stepped on, the wom...",Waiting to Exhale,"[based on novel, interracial relationship, sin...","[Whitney Houston, Angela Bassett, Loretta Devine]",[Forest Whitaker]
4,[Comedy],11862,Just when George Banks has recovered from his ...,Father of the Bride Part II,"[baby, midlife crisis, confidence, aging, daug...","[Steve Martin, Diane Keaton, Martin Short]",[Charles Shyer]


In [38]:
type(data['overview'][0])

str

In [39]:
def handle_data(lst):
    l = []
    for i in lst:
        l.append(i.lower().replace(" ", ""))
    return l

In [40]:
data['cast'] = data['cast'].apply(handle_data)

In [41]:
data['crew'] = data['crew'].apply(handle_data)

In [42]:
data['genres'] = data['genres'].apply(handle_data)

In [43]:
data['keywords'] = data['keywords'].apply(handle_data)


In [44]:
data['original_title'] = data['title']

In [45]:
data['title'] = data['title'].str.lower().str.replace(" ", "")

In [46]:
data['temp'] = data['title'].str.split(" ") + data['genres']*4 + data['keywords']*2 + data['cast'] + data['crew']*3

In [47]:
data['temp'][0]

['toystory',
 'animation',
 'comedy',
 'family',
 'animation',
 'comedy',
 'family',
 'animation',
 'comedy',
 'family',
 'animation',
 'comedy',
 'family',
 'jealousy',
 'toy',
 'boy',
 'friendship',
 'friends',
 'rivalry',
 'boynextdoor',
 'newtoy',
 'toycomestolife',
 'jealousy',
 'toy',
 'boy',
 'friendship',
 'friends',
 'rivalry',
 'boynextdoor',
 'newtoy',
 'toycomestolife',
 'tomhanks',
 'timallen',
 'donrickles',
 'johnlasseter',
 'johnlasseter',
 'johnlasseter']

In [48]:
data['overview'] = data['overview'].str.lower().str.split()


In [49]:
data['overview'][0]

['led',
 'by',
 'woody,',
 "andy's",
 'toys',
 'live',
 'happily',
 'in',
 'his',
 'room',
 'until',
 "andy's",
 'birthday',
 'brings',
 'buzz',
 'lightyear',
 'onto',
 'the',
 'scene.',
 'afraid',
 'of',
 'losing',
 'his',
 'place',
 'in',
 "andy's",
 'heart,',
 'woody',
 'plots',
 'against',
 'buzz.',
 'but',
 'when',
 'circumstances',
 'separate',
 'buzz',
 'and',
 'woody',
 'from',
 'their',
 'owner,',
 'the',
 'duo',
 'eventually',
 'learns',
 'to',
 'put',
 'aside',
 'their',
 'differences.']

In [50]:
# # removing punctuation 
# import string

# def remove_punc(lst):
#     table = str.maketrans('', '', string.punctuation)
#     result = []
    
#     for text in lst:
#         cleaned = text.translate(table)
        
#         # choose ONLY one version
#         final_text = cleaned if cleaned != text else text
        
#         # avoid duplicates
#         if final_text not in result:
#             result.append(final_text)
    
#     return result

# remove_punc(data['overview'][0])

In [51]:
import string
def remove_punc(lst):
    l = []
    for word in lst:
        cleaned = word
        for char in string.punctuation:
            if char in word:
                cleaned = cleaned.replace(char, '')
        if cleaned not in l:
            l.append(cleaned)
    return l

remove_punc(data['overview'][0])
        

['led',
 'by',
 'woody',
 'andys',
 'toys',
 'live',
 'happily',
 'in',
 'his',
 'room',
 'until',
 'birthday',
 'brings',
 'buzz',
 'lightyear',
 'onto',
 'the',
 'scene',
 'afraid',
 'of',
 'losing',
 'place',
 'heart',
 'plots',
 'against',
 'but',
 'when',
 'circumstances',
 'separate',
 'and',
 'from',
 'their',
 'owner',
 'duo',
 'eventually',
 'learns',
 'to',
 'put',
 'aside',
 'differences']

In [52]:
data['overview'] = data['overview'].apply(remove_punc)
data['overview']

0        [led, by, woody, andys, toys, live, happily, i...
1        [when, siblings, judy, and, peter, discover, a...
2        [a, family, wedding, reignites, the, ancient, ...
3        [cheated, on, mistreated, and, stepped, the, w...
4        [just, when, george, banks, has, recovered, fr...
                               ...                        
46618    [an, unsuccessful, sculptor, saves, a, madman,...
46619    [in, this, truecrime, documentary, we, delve, ...
46620    [a, film, archivist, revisits, the, story, of,...
46623       [rising, and, falling, between, a, man, woman]
46624    [an, artist, struggles, to, finish, his, work,...
Name: overview, Length: 30231, dtype: object

In [53]:
len(data['overview'][0])

40

In [54]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

en_stop = stopwords.words('english')

def remove_stopwords(lst):
    l = []
    for word in lst:
        if word in en_stop:
            pass
        else:
            l.append(word)

    return l

remove_stopwords(data['overview'][0])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dines\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['led',
 'woody',
 'andys',
 'toys',
 'live',
 'happily',
 'room',
 'birthday',
 'brings',
 'buzz',
 'lightyear',
 'onto',
 'scene',
 'afraid',
 'losing',
 'place',
 'heart',
 'plots',
 'circumstances',
 'separate',
 'owner',
 'duo',
 'eventually',
 'learns',
 'put',
 'aside',
 'differences']

In [55]:
data['overview'] = data['overview'].apply(remove_stopwords)
data['overview']

0        [led, woody, andys, toys, live, happily, room,...
1        [siblings, judy, peter, discover, enchanted, b...
2        [family, wedding, reignites, ancient, feud, ne...
3        [cheated, mistreated, stepped, women, holding,...
4        [george, banks, recovered, daughters, wedding,...
                               ...                        
46618    [unsuccessful, sculptor, saves, madman, named,...
46619    [truecrime, documentary, delve, murder, spree,...
46620    [film, archivist, revisits, story, rustin, par...
46623                        [rising, falling, man, woman]
46624    [artist, struggles, finish, work, storyline, c...
Name: overview, Length: 30231, dtype: object

In [56]:
len(data['overview'][0])

27

In [57]:
ps = PorterStemmer()
data['overview'] = data['overview'].apply(lambda words: [ps.stem(word) for word in words])

In [58]:
data['tags'] = data['temp'] + data['overview']

In [59]:
data['tags'][0]

['toystory',
 'animation',
 'comedy',
 'family',
 'animation',
 'comedy',
 'family',
 'animation',
 'comedy',
 'family',
 'animation',
 'comedy',
 'family',
 'jealousy',
 'toy',
 'boy',
 'friendship',
 'friends',
 'rivalry',
 'boynextdoor',
 'newtoy',
 'toycomestolife',
 'jealousy',
 'toy',
 'boy',
 'friendship',
 'friends',
 'rivalry',
 'boynextdoor',
 'newtoy',
 'toycomestolife',
 'tomhanks',
 'timallen',
 'donrickles',
 'johnlasseter',
 'johnlasseter',
 'johnlasseter',
 'led',
 'woodi',
 'andi',
 'toy',
 'live',
 'happili',
 'room',
 'birthday',
 'bring',
 'buzz',
 'lightyear',
 'onto',
 'scene',
 'afraid',
 'lose',
 'place',
 'heart',
 'plot',
 'circumst',
 'separ',
 'owner',
 'duo',
 'eventu',
 'learn',
 'put',
 'asid',
 'differ']

In [60]:
data.columns

Index(['genres', 'id', 'overview', 'title', 'keywords', 'cast', 'crew',
       'original_title', 'temp', 'tags'],
      dtype='object')

In [61]:
data.drop(columns = ['genres', 'overview', 'keywords', 'cast', 'crew', 'temp'], inplace = True)

In [62]:
data.head()

,id,title,original_title,tags
0,862,toystory,Toy Story,"[toystory, animation, comedy, family, animatio..."
1,8844,jumanji,Jumanji,"[jumanji, adventure, fantasy, family, adventur..."
2,15602,grumpieroldmen,Grumpier Old Men,"[grumpieroldmen, romance, comedy, romance, com..."
3,31357,waitingtoexhale,Waiting to Exhale,"[waitingtoexhale, comedy, drama, romance, come..."
4,11862,fatherofthebridepartii,Father of the Bride Part II,"[fatherofthebridepartii, comedy, comedy, comed..."


In [63]:
data['tags'] = data['tags'].apply(lambda x: " ".join(x))
data['tags']

0        toystory animation comedy family animation com...
1        jumanji adventure fantasy family adventure fan...
2        grumpieroldmen romance comedy romance comedy r...
3        waitingtoexhale comedy drama romance comedy dr...
4        fatherofthebridepartii comedy comedy comedy co...
                               ...                        
46618    houseofhorrors horror mystery thriller horror ...
46619    shadowoftheblairwitch mystery horror mystery h...
46620    theburkittsville7 horror horror horror horror ...
46623    subdue drama family drama family drama family ...
46624    centuryofbirthing drama drama drama drama arti...
Name: tags, Length: 30231, dtype: object

In [64]:
data.isnull().sum()

id                0
title             0
original_title    0
tags              0
dtype: int64

In [65]:
data.duplicated().sum()

602

In [66]:
data.drop_duplicates(inplace = True)

In [67]:
data.reset_index(drop = True, inplace = True)

In [68]:
TfIdf = TfidfVectorizer(max_features = 5000)
TfIdf

TfidfVectorizer(max_features=5000)

In [69]:
vectors = TfIdf.fit_transform(data['tags']).toarray()

In [70]:
vectors

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [71]:
len(vectors[0])

5000

In [72]:
similarity = cosine_similarity(vectors)

In [73]:
sorted(list(enumerate(similarity[0])),reverse = True, key = lambda x: x[1] )[1:6]

[(19350, 0.5286095338093264),
 (25152, 0.5061119984647422),
 (12673, 0.48893156184239805),
 (18565, 0.4734113040114584),
 (16961, 0.43870751986136625)]

In [101]:
def recommend(movie):
    movie_row = data[data['title'] == movie.lower().replace(" ", "")]

    if movie_row.empty:
        print("The movie you are requesting is not present")
        return
    else:
        movie_index = movie_row.index[0]
    distances = similarity[movie_index]
    movie_list = sorted(list(enumerate(distances)),reverse = True, key = lambda x: x[1] )[1:6]

    for i in movie_list:
        print(data.iloc[i[0]].original_title)

In [103]:
recommend('the Dark Knight')

The Dark Knight Rises
Batman: Under the Red Hood
Batman Begins
Batman: Assault on Arkham
Atom Man vs Superman


In [76]:
data['original_title'][:20]

0                          Toy Story
1                            Jumanji
2                   Grumpier Old Men
3                  Waiting to Exhale
4        Father of the Bride Part II
5                               Heat
6                            Sabrina
7                       Sudden Death
8                          GoldenEye
9             The American President
10       Dracula: Dead and Loving It
11                             Balto
12                             Nixon
13                  Cutthroat Island
14                            Casino
15             Sense and Sensibility
16                        Four Rooms
17    Ace Ventura: When Nature Calls
18                       Money Train
19                        Get Shorty
Name: original_title, dtype: object

In [108]:
import pickle
pickle.dump(data.to_dict(), open('data.pkl', mode = 'wb'))

In [109]:
pickle.dump(similarity, open('similarity.pkl', mode = 'wb'))